# Legacy vs H121 — OFA comparison over June 2020 (innov_newQC)

Compares Legacy BUFR ASCAT (species 9/10/11) against H121 CDR ASCAT (species 14/15/16) as written
to ObsFcstAna, using the full June 2020 `hsaf_cdr_test_DAv8_M36_202006_innov_newQC` run — the run
we're treating as the new default (see `innov_vs_innov_newqc.ipynb` for why).

July 2020 OFA data isn't downloaded yet, so this notebook uses the full June window instead
(2020-06-01 through 2020-06-30; June 1 and June 30 are partial days at the run's edges).

Both products come from the *same* run/OFA files here — this is a legacy-vs-H121 product
comparison, not a QC-variant comparison.


In [ ]:
import sys, os
from pathlib import Path


def _find_lib_root():
    cwd = Path(os.path.abspath(''))
    for p in [cwd] + list(cwd.parents):
        if (p / 'lib').exists() and (p / 'lib' / 'readers.py').exists():
            return p
        for child in p.glob('projects/*/lib'):
            if (child / 'readers.py').exists():
                return child.parent
    raise RuntimeError(f'Cannot find ascat_da/lib/ from {cwd}')


_root = _find_lib_root()
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

_repo_root = Path(_root).parents[1]
_common_io = _repo_root / 'common' / 'python' / 'io'
if str(_common_io) not in sys.path:
    sys.path.insert(0, str(_common_io))
from read_GEOSldas import read_tilecoord

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature


In [ ]:
# ── Configuration — edit here ─────────────────────────────────────────────────
START_DATE = '2020-06-01'
END_DATE   = '2020-06-30'

CACHE_VERSION = 'newqc'
CACHE_DIR = Path(_root) / '.cache' / 'ofa'
CACHE_TAG = f"{START_DATE.replace('-', '')}_{END_DATE.replace('-', '')}_{CACHE_VERSION}"

PRODUCTS = {
    'legacy': {'Metop-A': 9,  'Metop-B': 10, 'Metop-C': 11},
    'h121':   {'Metop-A': 14, 'Metop-B': 15, 'Metop-C': 16},
}
PLATFORM_COLOR = {'Metop-A': '#1f77b4', 'Metop-B': '#ff7f0e', 'Metop-C': '#2ca02c'}

# Tile lat/lon must come from the tilecoord file, not the OFA file's own lat/lon
# (the latter is the per-cycle super-ob center and jitters slightly within a tile).
TILE_BASE = '/Users/amfox/Desktop/ASCAT_SSM_CDR/discover_sample/tilecoord/hsaf_cdr_test_DAv8_M36_202006_innov'
TILECOORD = f'{TILE_BASE}.ldas_tilecoord.bin'

print(f"Date range: {START_DATE} .. {END_DATE}")
print(f"Cache tag: {CACHE_TAG}")


## 1. Load cached OFA tile/cycle table

Pre-aggregated with `build_ofa_cache.py` (one row per date/cycle/species/tile):

```bash
python projects/ascat_da/scripts/build_ofa_cache.py \
  --start-date 2020-06-01 --end-date 2020-06-30 \
  --ofa-dir data/hsaf_cdr_test/hsaf_cdr_test_DAv8_M36_202006_innov_newQC/output/SMAP_EASEv2_M36_GLOBAL/ana/ens_avg/Y2020/M06 \
  --out-dir projects/ascat_da/.cache/ofa \
  --version newqc
```


In [ ]:
pkl = CACHE_DIR / f'ofa_ascat_tile_cycle_{CACHE_TAG}.pkl'
if not pkl.exists():
    raise FileNotFoundError(f'Missing OFA cache: {pkl}\nRun build_ofa_cache.py (see markdown above).')

ofa = pd.read_pickle(pkl)
print(f"{len(ofa):,} tile/cycle rows, {ofa['date'].min()} .. {ofa['date'].max()}")

tile_coord = read_tilecoord(TILECOORD)
tile_latlon = pd.DataFrame({
    'tilenum': tile_coord['tile_id'].astype('int64'),
    'tile_lat': tile_coord['com_lat'],
    'tile_lon': tile_coord['com_lon'],
}).drop_duplicates('tilenum')

ofa = ofa.merge(tile_latlon, on='tilenum', how='left')
n_missing = ofa['tile_lat'].isna().sum()
if n_missing:
    print(f"WARNING: {n_missing} OFA rows have no matching tilecoord entry")


## 2. Observation counts: legacy vs H121

Raw super-ob counts by product/platform over the full month.


In [ ]:
counts = (
    ofa.groupby(['product', 'platform'], as_index=False)
    .agg(total_ofa_obs=('tilenum', 'size'))
)
pivot = counts.pivot_table(index='platform', columns='product', values='total_ofa_obs')
pivot['h121_to_legacy_ratio'] = pivot['h121'] / pivot['legacy']
display(pivot.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(pivot))
width = 0.35
ax.bar(x - width / 2, pivot['legacy'], width, label='legacy', color='#555555')
ax.bar(x + width / 2, pivot['h121'], width, label='h121', color='#2ca02c')
ax.set_xticks(x)
ax.set_xticklabels(pivot.index)
ax.set_ylabel('Total OFA obs (June 2020)')
ax.set_title('Obs counts: legacy vs H121')
ax.legend()
fig.tight_layout()


## 3. Maps: per-tile obs counts (legacy vs H121)

Per-tile obs counts summed over June, by platform (columns). Rows: legacy, H121, and H121-minus-legacy.


In [ ]:
platforms = ['Metop-A', 'Metop-B', 'Metop-C']


def tile_counts(df, species_id):
    sub = df[df['species'] == species_id]
    return (
        sub.groupby('tilenum', as_index=False)
        .agg(lat=('tile_lat', 'first'), lon=('tile_lon', 'first'), n_obs=('tilenum', 'size'))
    )


tile_maps = {}
for plat in platforms:
    legacy_t = tile_counts(ofa, PRODUCTS['legacy'][plat])
    h121_t = tile_counts(ofa, PRODUCTS['h121'][plat])
    merged = legacy_t.merge(h121_t, on='tilenum', how='outer', suffixes=('_legacy', '_h121'))
    merged['lat'] = merged['lat_legacy'].combine_first(merged['lat_h121'])
    merged['lon'] = merged['lon_legacy'].combine_first(merged['lon_h121'])
    merged['n_obs_legacy'] = merged['n_obs_legacy'].fillna(0)
    merged['n_obs_h121'] = merged['n_obs_h121'].fillna(0)
    merged['diff'] = merged['n_obs_h121'] - merged['n_obs_legacy']
    tile_maps[plat] = merged

from matplotlib.colors import LogNorm, TwoSlopeNorm

count_norm = LogNorm(vmin=1, vmax=max(m[['n_obs_legacy', 'n_obs_h121']].values.max() for m in tile_maps.values()))
diff_abs_max = max(m['diff'].abs().max() for m in tile_maps.values())
diff_norm = TwoSlopeNorm(vmin=-diff_abs_max, vcenter=0, vmax=diff_abs_max)

fig, axes = plt.subplots(3, 3, figsize=(15, 10), subplot_kw={'projection': ccrs.Robinson()})
row_specs = [
    ('n_obs_legacy', 'legacy', 'viridis', count_norm),
    ('n_obs_h121', 'h121', 'viridis', count_norm),
    ('diff', 'h121 - legacy', 'RdBu_r', diff_norm),
]

for row, (col, row_label, cmap, norm) in enumerate(row_specs):
    for ax_col, plat in enumerate(platforms):
        ax = axes[row, ax_col]
        m = tile_maps[plat]
        sc = ax.scatter(
            m['lon'], m['lat'], c=m[col], s=0.5, cmap=cmap, norm=norm,
            transform=ccrs.PlateCarree(),
        )
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
        ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
        if row == 0:
            ax.set_title(plat, fontsize=11)
        if ax_col == 0:
            ax.text(-0.08, 0.5, row_label, transform=ax.transAxes, rotation=90,
                     va='center', ha='center', fontsize=10)
    fig.colorbar(sc, ax=axes[row, :].tolist(), shrink=0.7, pad=0.01,
                 label='obs count' if row < 2 else 'Δ obs count')

fig.suptitle('Legacy vs H121 obs count per tile (June 2020 total)', y=0.99)


In [ ]:
def tile_value_map(df, species_id, value_col, agg='mean'):
    sub = df[df['species'] == species_id].copy()
    if agg == 'mean_abs':
        sub['_val'] = sub[value_col].abs()
        agg_func = 'mean'
    else:
        sub['_val'] = sub[value_col]
        agg_func = agg
    return (
        sub.groupby('tilenum', as_index=False)
        .agg(lat=('tile_lat', 'first'), lon=('tile_lon', 'first'), value=('_val', agg_func))
    )


def plot_legacy_h121_diff_maps(value_col, agg, title, cbar_label, diff_label='h121 - legacy'):
    tmaps = {}
    for plat in platforms:
        legacy_t = tile_value_map(ofa, PRODUCTS['legacy'][plat], value_col, agg)
        h121_t = tile_value_map(ofa, PRODUCTS['h121'][plat], value_col, agg)
        merged = legacy_t.merge(h121_t, on='tilenum', how='outer', suffixes=('_legacy', '_h121'))
        merged['lat'] = merged['lat_legacy'].combine_first(merged['lat_h121'])
        merged['lon'] = merged['lon_legacy'].combine_first(merged['lon_h121'])
        merged['diff'] = merged['value_h121'] - merged['value_legacy']
        tmaps[plat] = merged

    vmin = min(m[['value_legacy', 'value_h121']].min().min() for m in tmaps.values())
    vmax = max(m[['value_legacy', 'value_h121']].max().max() for m in tmaps.values())
    diff_abs_max = max(m['diff'].abs().max() for m in tmaps.values())
    diff_norm = TwoSlopeNorm(vmin=-diff_abs_max, vcenter=0, vmax=diff_abs_max)

    fig, axes = plt.subplots(3, 3, figsize=(15, 10), subplot_kw={'projection': ccrs.Robinson()})
    row_specs = [
        ('value_legacy', 'legacy', 'viridis', dict(vmin=vmin, vmax=vmax)),
        ('value_h121', 'h121', 'viridis', dict(vmin=vmin, vmax=vmax)),
        ('diff', diff_label, 'RdBu_r', dict(norm=diff_norm)),
    ]
    for row, (col, row_label, cmap, norm_kw) in enumerate(row_specs):
        for ax_col, plat in enumerate(platforms):
            ax = axes[row, ax_col]
            m = tmaps[plat]
            sc = ax.scatter(
                m['lon'], m['lat'], c=m[col], s=0.5, cmap=cmap,
                transform=ccrs.PlateCarree(), **norm_kw,
            )
            ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
            ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
            if row == 0:
                ax.set_title(plat, fontsize=11)
            if ax_col == 0:
                ax.text(-0.08, 0.5, row_label, transform=ax.transAxes, rotation=90,
                         va='center', ha='center', fontsize=10)
        fig.colorbar(sc, ax=axes[row, :].tolist(), shrink=0.7, pad=0.01,
                     label=cbar_label if row < 2 else f'Δ {cbar_label}')
    fig.suptitle(title, y=0.99)
    return tmaps


## 4. Maps: mean obs value (legacy vs H121)

Per-tile mean obs over June (degree-of-saturation %, model-space units), rows = legacy /
H121 / diff, columns = Metop-A/B/C.


In [ ]:
_ = plot_legacy_h121_diff_maps(
    'obs_pct', 'mean', 'Mean obs per tile (June 2020): legacy vs H121', 'mean obs',
)


## 5. Maps: mean |innovation| (legacy vs H121)

Per-tile mean absolute innov (mean(|obs - fcst|)) over June — a measure of typical
innovation magnitude, independent of sign cancellation.


In [ ]:
_ = plot_legacy_h121_diff_maps(
    'innov_pct', 'mean_abs', 'Mean |innov| per tile (June 2020): legacy vs H121',
    'mean |innov|', diff_label='h121 - legacy (|innov|)',
)


## 6. Coverage fraction (resolution-agnostic)

As in `innov_vs_innov_newqc.ipynb`: raw counts aren't directly comparable across products with
different swath/footprint resolutions. Coverage fraction = share of June's analysis cycles
(30 days x 8 cycles = 240) where *any* platform of that product produced a super-ob for the tile.


In [ ]:
N_CYCLES_TOTAL = len(pd.date_range(START_DATE, END_DATE)) * 8


def coverage_fraction(df, species_list, n_total):
    sub = (
        df[df['species'].isin(species_list)][['tilenum', 'date', 'cycle', 'tile_lat', 'tile_lon']]
        .drop_duplicates(['tilenum', 'date', 'cycle'])
    )
    cov = sub.groupby('tilenum').agg(
        n_cycles=('date', 'size'), lat=('tile_lat', 'first'), lon=('tile_lon', 'first'),
    )
    cov['coverage_frac'] = cov['n_cycles'] / n_total
    return cov.reset_index()


legacy_species = list(PRODUCTS['legacy'].values())
h121_species = list(PRODUCTS['h121'].values())

cov_legacy = coverage_fraction(ofa, legacy_species, N_CYCLES_TOTAL)
cov_h121 = coverage_fraction(ofa, h121_species, N_CYCLES_TOTAL)

cov_panels = [
    ('Legacy (Metop-A/B/C)', cov_legacy),
    ('H121 (Metop-A/B/C)', cov_h121),
]

fig, axes = plt.subplots(2, 1, figsize=(22, 10), subplot_kw={'projection': ccrs.Robinson()})
for ax, (title, cov) in zip(axes, cov_panels):
    sc = ax.scatter(
        cov['lon'], cov['lat'], c=cov['coverage_frac'], s=0.5, cmap='viridis',
        vmin=0, vmax=1, transform=ccrs.PlateCarree(),
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
    ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
    ax.set_title(f"{title}  (mean={cov['coverage_frac'].mean():.2f})", fontsize=11)
fig.colorbar(sc, ax=axes.tolist(), shrink=0.7, pad=0.02, label='fraction of cycles with a super-ob')
fig.suptitle('Per-tile coverage fraction — June 2020', y=1.01)


## 7. Distribution comparison: obs and innov

Legacy and H121 obs are both in degree-of-saturation % (model-space units after scaling), so
direct distributional comparison is meaningful here — unlike the raw counts above.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex='row')
for col, plat in enumerate(platforms):
    legacy_obs = ofa.loc[ofa['species'] == PRODUCTS['legacy'][plat], 'obs_pct'].dropna()
    h121_obs = ofa.loc[ofa['species'] == PRODUCTS['h121'][plat], 'obs_pct'].dropna()
    axes[0, col].hist(legacy_obs, bins=60, alpha=0.6, label='legacy', color='#555555', density=True)
    axes[0, col].hist(h121_obs, bins=60, alpha=0.6, label='h121', color='#2ca02c', density=True)
    axes[0, col].set_title(f'{plat}: obs')

    legacy_innov = ofa.loc[ofa['species'] == PRODUCTS['legacy'][plat], 'innov_pct'].dropna()
    h121_innov = ofa.loc[ofa['species'] == PRODUCTS['h121'][plat], 'innov_pct'].dropna()
    axes[1, col].hist(legacy_innov, bins=60, alpha=0.6, label='legacy', color='#555555', density=True)
    axes[1, col].hist(h121_innov, bins=60, alpha=0.6, label='h121', color='#2ca02c', density=True)
    axes[1, col].set_title(f'{plat}: innov')

axes[0, 0].legend()
fig.tight_layout()


## 8. Matched scatter: legacy vs H121 obs values

Match legacy and H121 super-obs on (date, cycle, tilenum) per platform — same satellite, same
analysis window, same model tile — so the comparison is of the actual observation value, not
just aggregate distributions. Only tile-cycles where *both* products reported a super-ob are kept.


In [ ]:
def _scatter_stats(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if len(x) == 0:
        return dict(n=0, bias=np.nan, rmse=np.nan, r=np.nan)
    diff = y - x
    return dict(
        n=len(x),
        bias=float(np.mean(diff)),
        rmse=float(np.sqrt(np.mean(diff ** 2))),
        r=float(np.corrcoef(x, y)[0, 1]) if len(x) > 1 else np.nan,
    )


def _sample_for_plot(df, max_points=80000, random_state=42):
    if len(df) <= max_points:
        return df
    return df.sample(max_points, random_state=random_state)


MATCH_KEYS = ['date', 'cycle', 'tilenum']

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
matched_obs = {}
for ax, plat in zip(axes, platforms):
    legacy_sub = ofa.loc[ofa['species'] == PRODUCTS['legacy'][plat], MATCH_KEYS + ['obs_pct']]
    h121_sub = ofa.loc[ofa['species'] == PRODUCTS['h121'][plat], MATCH_KEYS + ['obs_pct']]
    matched = legacy_sub.merge(h121_sub, on=MATCH_KEYS, suffixes=('_legacy', '_h121'))
    matched = matched.merge(tile_latlon[['tilenum', 'tile_lat']], on='tilenum', how='left')
    matched_obs[plat] = matched

    stats = _scatter_stats(matched['obs_pct_legacy'], matched['obs_pct_h121'])
    plot_df = _sample_for_plot(matched)
    sc = ax.scatter(
        plot_df['obs_pct_legacy'], plot_df['obs_pct_h121'], s=2, alpha=0.3,
        c=plot_df['tile_lat'], cmap='coolwarm', vmin=-60, vmax=85,
    )
    ax.plot([0, 100], [0, 100], 'k--', linewidth=1)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_xlabel('legacy obs')
    ax.set_ylabel('h121 obs')
    ax.set_title(
        f"{plat}  (n={stats['n']:,})\n"
        f"bias={stats['bias']:.2f}  rmse={stats['rmse']:.2f}  r={stats['r']:.2f}"
    )

fig.tight_layout(rect=[0.04, 0, 1, 1])
cax = fig.add_axes([0.01, 0.15, 0.012, 0.7])
fig.colorbar(sc, cax=cax, label='tile latitude')
fig.suptitle('Matched legacy vs H121 obs values (same date/cycle/tile) — June 2020', y=1.03)


## 9. Maps: goodness of fit (matched legacy vs H121)

Per-tile fit statistics computed from the matched pairs in Section 8: correlation (r), bias
(h121 - legacy), and RMSE. Tiles with fewer than `MIN_MATCHED_N` matched cycles are masked out —
per-tile r is unreliable on small samples — and shown separately as a sample-count map so the
gaps are visible rather than silently dropped.


In [ ]:
MIN_MATCHED_N = 10


def tile_fit_stats(matched, min_n=MIN_MATCHED_N):
    def _stats(g):
        x = g['obs_pct_legacy'].to_numpy(float)
        y = g['obs_pct_h121'].to_numpy(float)
        n = len(g)
        diff = y - x
        r = np.corrcoef(x, y)[0, 1] if n >= 2 and np.std(x) > 0 and np.std(y) > 0 else np.nan
        return pd.Series({
            'n': n,
            'r': r,
            'bias': diff.mean(),
            'rmse': np.sqrt((diff ** 2).mean()),
        })

    stats = matched.groupby('tilenum').apply(_stats, include_groups=False).reset_index()
    stats = stats.merge(tile_latlon, on='tilenum', how='left')
    stats.loc[stats['n'] < min_n, ['r', 'bias', 'rmse']] = np.nan
    return stats


fit_stats = {plat: tile_fit_stats(matched_obs[plat]) for plat in platforms}

fig, axes = plt.subplots(3, 3, figsize=(15, 10), subplot_kw={'projection': ccrs.Robinson()})
row_specs = [
    ('r', 'correlation (r)', 'viridis', dict(vmin=-1, vmax=1)),
    ('bias', 'bias (h121-legacy)', 'RdBu_r', dict(vmin=-20, vmax=20)),
    ('rmse', 'RMSE', 'magma', dict(vmin=0, vmax=30)),
]
for row, (col, row_label, cmap, norm_kw) in enumerate(row_specs):
    for ax_col, plat in enumerate(platforms):
        ax = axes[row, ax_col]
        s = fit_stats[plat]
        sc = ax.scatter(
            s['tile_lon'], s['tile_lat'], c=s[col], s=0.5, cmap=cmap,
            transform=ccrs.PlateCarree(), **norm_kw,
        )
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
        ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
        if row == 0:
            ax.set_title(plat, fontsize=11)
        if ax_col == 0:
            ax.text(-0.08, 0.5, row_label, transform=ax.transAxes, rotation=90,
                     va='center', ha='center', fontsize=10)
    fig.colorbar(sc, ax=axes[row, :].tolist(), shrink=0.7, pad=0.01, label=row_label)
fig.suptitle(f'Matched legacy vs H121 goodness of fit per tile (min n={MIN_MATCHED_N})', y=0.99)


Sample-count map (matched cycles per tile) — for context on where the fit stats above are
well- vs poorly-sampled.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), subplot_kw={'projection': ccrs.Robinson()})
n_max = max(s['n'].max() for s in fit_stats.values())
for ax, plat in zip(axes, platforms):
    s = fit_stats[plat]
    sc = ax.scatter(
        s['tile_lon'], s['tile_lat'], c=s['n'], s=0.5, cmap='viridis',
        norm=LogNorm(vmin=1, vmax=n_max), transform=ccrs.PlateCarree(),
    )
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
    ax.set_extent([-180, 180, -60, 85], crs=ccrs.PlateCarree())
    ax.set_title(plat, fontsize=11)
fig.colorbar(sc, ax=axes.tolist(), shrink=0.7, pad=0.02, label='matched cycles per tile')
fig.suptitle('Matched-pair sample count per tile', y=1.03)


## 10. Daily obs counts over June

Daily total obs by product — checks for swath/orbit-related dips or trends within the month,
separate from the QC-driven shifts already characterized in `innov_vs_innov_newqc.ipynb`.


In [ ]:
daily = (
    ofa.groupby(['date', 'product'], as_index=False)
    .agg(n_obs=('tilenum', 'size'))
    .pivot(index='date', columns='product', values='n_obs')
)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily.index, daily['legacy'], label='legacy', color='#555555', marker='o', markersize=3)
ax.plot(daily.index, daily['h121'], label='h121', color='#2ca02c', marker='o', markersize=3)
ax.set_ylabel('Daily obs count (all platforms)')
ax.set_title('Daily obs counts: legacy vs H121 — June 2020')
ax.tick_params(axis='x', rotation=45)
ax.legend()
fig.tight_layout()


## 11. Summary

In [ ]:
summary_rows = []
for plat in platforms:
    legacy_n = int(pivot.loc[plat, 'legacy'])
    h121_n = int(pivot.loc[plat, 'h121'])
    summary_rows.append({
        'platform': plat,
        'legacy_obs': legacy_n,
        'h121_obs': h121_n,
        'h121_to_legacy_ratio': round(h121_n / legacy_n, 2),
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
print(f"Mean coverage fraction: legacy={cov_legacy['coverage_frac'].mean():.2f}, "
      f"h121={cov_h121['coverage_frac'].mean():.2f}")
